# 03 — PyArrow: Columnar Memory, Parquet, Zero-Copy Interop

**Overview:** Arrow's columnar in-memory `Table`, writing/reading Parquet, and how Pandas and
Polars both sit on top of (or can convert to/from) the same Arrow format.

**Required libraries:** `pyarrow`, `pandas`, `polars`.

```
!pip install -q pyarrow   # not preinstalled on Colab
```

Connects to: `01_pandas.ipynb` / `02_polars.ipynb` (same dataset). Leads to: `04_datasets`
(Hugging Face `datasets` is itself built on Arrow).

In [1]:
%pip install -q polars pyarrow

Note: you may need to restart the kernel to use updated packages.


## 0. Environment & data

In [2]:
import sys
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
import polars as pl

print("Python:", sys.version.split()[0])
print("pyarrow:", pa.__version__)


DATA_PATH = Path("data/support_tickets.csv")

def generate_support_tickets_csv(path, n=300, seed=42):
    """Self-contained, deterministic synthetic dataset -- no external script or
    download needed, so this cell works the same in a fresh Colab runtime as it
    does locally."""
    import csv, random
    rng = random.Random(seed)
    categories = ["billing", "login", "performance", "bug", None]
    priorities = ["low", "medium", "high"]
    rows = []
    for i in range(n):
        rows.append({
            "ticket_id": i,
            "user_id": rng.randint(1, 50),
            "category": rng.choice(categories),
            "priority": rng.choice(priorities),
            "resolved": rng.choice([True, False]),
            "response_time_minutes": round(rng.uniform(5, 500), 1),
        })
    rows.extend(rows[:10])   # deterministic duplicates, used later to demo dedup handling
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

if not DATA_PATH.exists():
    generate_support_tickets_csv(DATA_PATH)

pdf = pd.read_csv(DATA_PATH)
print("pandas shape:", pdf.shape)

Python: 3.12.5
pyarrow: 25.0.1
pandas shape: (310, 6)


## 1. Columnar vs row-oriented layout

💡 **Key Idea:** a row-oriented format (CSV, a Python list of dicts) stores each record
contiguously. A columnar format (Arrow, Parquet) stores each *column* contiguously — reading
just 2 of 20 columns only touches those 2 columns' bytes.

In [3]:
table = pa.Table.from_pandas(pdf)
print(table.schema)
print("num columns:", table.num_columns, "num rows:", table.num_rows)

assert table.num_rows == pdf.shape[0]
assert table.num_columns == pdf.shape[1]

# Columnar access: pull a single column without touching the others.
response_times = table.column("response_time_minutes")
print("\ncolumn type:", type(response_times), "length:", len(response_times))


ticket_id: int64
user_id: int64
category: large_string
priority: large_string
resolved: bool
response_time_minutes: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 980
num columns: 6 num rows: 310

column type: <class 'pyarrow.lib.ChunkedArray'> length: 310


## 2. Zero-copy interop between Pandas, Polars, and Arrow

In [4]:
# Pandas -> Arrow -> Polars, without re-parsing the source data.
pl_from_arrow = pl.from_arrow(table)
print("polars shape (from Arrow table):", pl_from_arrow.shape)
assert pl_from_arrow.shape == (pdf.shape[0], pdf.shape[1])

# Polars -> Arrow directly.
pl_df = pl.from_pandas(pdf)
arrow_from_polars = pl_df.to_arrow()
print("arrow table (from Polars) num_rows:", arrow_from_polars.num_rows)
assert arrow_from_polars.num_rows == pdf.shape[0]


polars shape (from Arrow table): (310, 6)
arrow table (from Polars) num_rows: 310


## 3. Parquet round-trip

🎯 **Interview Tip:** Parquet is the standard on-disk columnar format for ML datasets (what
Hugging Face `datasets` caches to, what most data lakes store). It's compressed and lets you
read a subset of columns without scanning the whole file.

In [5]:
PARQUET_PATH = Path("data/support_tickets.parquet")
pq.write_table(table, PARQUET_PATH)

table_read_back = pq.read_table(PARQUET_PATH)
pdf_read_back = table_read_back.to_pandas()

print("round-trip shapes match:", pdf_read_back.shape == pdf.shape)
pd.testing.assert_frame_equal(
    pdf_read_back.sort_values("ticket_id").reset_index(drop=True),
    pdf.sort_values("ticket_id").reset_index(drop=True),
)
print("round-trip data verified identical.")


round-trip shapes match: True
round-trip data verified identical.


🔬 **Experiment:** compare on-disk size of CSV vs Parquet for the same data.

In [6]:
csv_size = DATA_PATH.stat().st_size
parquet_size = PARQUET_PATH.stat().st_size

print(f"CSV size:     {csv_size:,} bytes")
print(f"Parquet size: {parquet_size:,} bytes")
print(f"Parquet is {csv_size / parquet_size:.2f}x smaller on this small dataset "
      "(the gap grows with data size and repetition in column values).")


CSV size:     9,402 bytes
Parquet size: 8,395 bytes
Parquet is 1.12x smaller on this small dataset (the gap grows with data size and repetition in column values).


## 4. Reading only a subset of columns

💡 **Key Idea:** this is the practical payoff of columnar storage — pulling 2 columns out of 6
from a Parquet file only reads those columns' bytes, unlike CSV which must scan every line.

In [7]:
subset = pq.read_table(PARQUET_PATH, columns=["ticket_id", "priority"])
print("subset schema:", subset.schema)
print("subset shape:", subset.num_rows, "rows x", subset.num_columns, "columns")
assert subset.num_columns == 2


subset schema: ticket_id: int64
priority: large_string
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 980
subset shape: 310 rows x 2 columns


## 5. Connection to what's next

Hugging Face `datasets` stores data in Arrow under the hood — every `Dataset` you load with
`load_dataset` or build with `Dataset.from_pandas` is an Arrow table with a friendlier API on
top. That's exactly what `04_datasets` covers next.

## 6. Interview questions
1. What's the practical difference between row-oriented and columnar storage?
2. Why can Parquet be both smaller on disk and faster to partially read than CSV?
3. What makes Arrow a good interchange format between Pandas, Polars, and Hugging Face `datasets`?
4. Why would you read only specific columns from a Parquet file instead of the whole table?
5. What does "zero-copy" mean in the context of converting between Polars and Arrow?

## 7. Exercises
1. Write the cleaned (deduplicated) DataFrame from `01_pandas.ipynb` to Parquet and verify a round-trip.
2. Compare file sizes of the same data saved as CSV, JSON, and Parquet.
3. Read back only the `category` and `resolved` columns from the Parquet file and compute the resolved rate per category without loading any other column.

## 8. Summary

✅ Arrow's columnar format is the shared substrate underneath Pandas, Polars, and Hugging Face
`datasets`; Parquet is its standard on-disk form. This closes out `03_data_libraries` — next:
`04_datasets` for dataset design and Hugging Face `datasets`.